# 실습 미션 1 · 사내 챗봇에 마스킹, 감사 로그, 모니터링 적용하기

가온반도체(가명) 총무팀은 출입과 보안 규정을 안내하는 사내 챗봇을 운영하고 있습니다.

사용자가 질문하면 챗봇이 사내 문서를 검색하고, 검색한 문서와 질문으로 프롬프트를 만들어 Claude 에 보냅니다. 직원과 협력사 직원이 함께 쓰며, 문서 검색에는 사용자가 볼 수 있는 문서만 찾는 권한 필터가 이미 적용되어 있습니다.

<br>

이번 미션에서는 이 챗봇에서 개인정보가 새지 않도록 세 가지를 직접 설정합니다.

- **마스킹** · Claude 에 보내기 전에 개인정보를 가립니다
- **감사 로그** · 챗봇이 처리한 요청을 기록합니다
- **모니터링** · 로그를 사용자 계정별로 모아, 이상 기록이 있는 사용자를 찾습니다

<br>

**[챗봇이 답하는 순서]** 굵게 표시한 부분이 이번 미션에서 설정하는 곳입니다.

질문 → 문서 검색 → 프롬프트 만들기(사용자 구분, 검색한 문서, 질문) → **① 마스킹** → Claude → 답 복원 → **② 감사 로그** → **③ 모니터링**

<br>

**[재료]** `mission_data` 폴더에 있으며, 모두 가상 자료입니다.

- `documents.json`: 챗봇이 검색하는 문서 10건. 문서마다 볼 수 있는 사용자(직원, 협력사)가 적혀 있습니다
- `directory.json`: 사내 명부. 인사 DB 의 직원 이름과 어제까지의 방문 등록 명단입니다
- `today_visitors.json`: 오늘 아침 방문 등록 시스템에 새로 올라온 방문객 이름
- `public_contacts.json`: 가리지 않을 팀 대표 연락처
- `labeled_sentences.json`: 가려야 할 값을 표시해 둔 문장 9개 (마스킹 측정용)
- `day_requests.json`: 오늘 하루 챗봇에 들어온 요청 25건
- `haiku_cache.json`: 강사 환경에서 받아 둔 Claude 답. API 키가 없을 때 씁니다

<br>

**[진행 순서]**

| 단계 | 하는 일 |
| --- | --- |
| 1 | 마스킹: 보낸 프롬프트 확인 → 마스킹 적용 → 정답 표시 문장으로 측정 → 고치고 다시 측정 |
| 2 | 감사 로그: 기록할 항목을 정하고 하루치 요청을 기록 |
| 3 | 모니터링: 로그를 계정별로 모으고, 따로 확인할 계정 정하기 |

## 준비

코랩 왼쪽 열쇠 아이콘(보안 비밀)에 `ANTHROPIC_API_KEY` 가 있으면 Claude 를 직접 부르고, 없으면 받아 둔 답(`haiku_cache.json`)을 씁니다.

In [ ]:
%pip install -q anthropic gdown

In [ ]:
import os, json, re, hashlib, zipfile
from collections import defaultdict
import gdown, anthropic

if not os.path.exists('mission_data'):
    gdown.download(id='11GSaV5KITXSls5rbGpW2v1sfEsNeVuZP', output='day09_mission1_data.zip', quiet=True)
    zipfile.ZipFile('day09_mission1_data.zip').extractall('.')


def load(name):
    return json.load(open(f'mission_data/{name}', encoding='utf-8'))


documents = load('documents.json')
directory = load('directory.json')
today_visitors = load('today_visitors.json')
public_contacts = load('public_contacts.json')
labeled = load('labeled_sentences.json')
day_requests = load('day_requests.json')
CACHE = load('haiku_cache.json')

try:
    from google.colab import userdata
    API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    API_KEY = os.environ.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=API_KEY) if API_KEY else None
MODEL = 'claude-haiku-5-5'
print('Claude 직접 호출' if client else '받아 둔 답 사용 (API 키 없음)', f'· 문서 {len(documents)}건, 오늘 요청 {len(day_requests)}건')

In [ ]:
SYSTEM = ('너는 가온반도체 총무팀의 사내 안내 챗봇이다. 검색된 사내 문서에 근거해 한국어로 세 문장 안에 답한다. '
          '근거를 못 찾으면 못 찾았다고 답한다. 질문에 [이름1], [전화1] 같은 표시가 있으면 그 표시를 그대로 써서 답한다.')


def two_letter_pieces(text):
    """띄어쓰기를 지운 글을 두 글자씩 겹쳐 자른 조각 집합 (검색 점수 계산용)"""
    squeezed = re.sub(r'\s+', '', text)
    return {squeezed[i:i + 2] for i in range(len(squeezed) - 1)}


def search(question, user, k=3):
    """사용자가 볼 수 있는 문서 안에서 질문과 겹치는 조각이 많은 문서 k 건을 찾습니다 (5일차의 검색 전 권한 필터)"""
    pieces = two_letter_pieces(question)
    pool = [doc for doc in documents if user in doc['access']]
    scored = sorted(pool, key=lambda doc: -len(pieces & two_letter_pieces(doc['title'] + doc['text'])))
    return [doc for doc in scored[:k] if pieces & two_letter_pieces(doc['title'] + doc['text'])]


def build_prompt(question, user, hits):
    """사용자 구분, 검색한 문서, 질문으로 Claude 에 보낼 프롬프트를 만듭니다"""
    context = '\n\n'.join(f"[{doc['doc_id']}] {doc['title']}\n{doc['text']}" for doc in hits)
    return f'[사용자 구분] {user} 계정\n\n[검색된 문서]\n{context}\n\n[질문]\n{question}'


def call_claude(prompt):
    """Claude 를 부릅니다. API 키가 없으면 받아 둔 답을 꺼냅니다"""
    key = hashlib.md5((SYSTEM + prompt).encode()).hexdigest()
    if client:
        response = client.messages.create(model=MODEL, max_tokens=600, system=SYSTEM, thinking={'type': 'disabled'},
                                          messages=[{'role': 'user', 'content': prompt}])
        CACHE[key] = ''.join(block.text for block in response.content if block.type == 'text')   # 답 글만 모읍니다
        return CACHE[key]
    return CACHE.get(key, '(받아 둔 답이 없습니다. 바꾼 값으로 실행하려면 API 키가 필요합니다)')


def brief(text, limit):
    """줄바꿈을 지우고 limit 글자까지만 보여 줍니다"""
    text = ' '.join(text.split())
    return text if len(text) <= limit else text[:limit] + ' …'


def print_box(title, text):
    print('  ┌─ ' + title + ' ' + '─' * (70 - len(title)))
    for line in text.split('\n'):
        print('  │ ' + line)
    print('  └' + '─' * 74)


print('준비 완료')

## 1단계 · 마스킹 (예제 1-5)

### 1-1 · Claude 에 보낸 프롬프트 확인하기

오늘 들어온 요청 하나를 마스킹 없이 처리해 봅시다. 답만 보면 문제가 없어 보이지만, 그 답을 만들려고 **Claude 에 실제로 보낸 프롬프트**에 무엇이 들어 있었는지 확인해 봅시다.

In [ ]:
request = day_requests[1]
hits = search(request['question'], request['user'])
prompt = build_prompt(request['question'], request['user'], hits)
answer = call_claude(prompt)

print('질문 │', request['question'])
print('답   │', brief(answer, 300))
print_box('Claude 에 보낸 프롬프트', prompt)
print('프롬프트에 들어간 개인정보 │', ', '.join(value for value in request['personal'] if value in prompt))

### 1-2 · 마스킹 적용하기

이 챗봇의 마스킹은 아래와 같이 설계되어 있습니다.

Claude 에 보낼 프롬프트 전체(질문과 검색한 문서)에서 개인정보를 찾아 `[이름1]`, `[전화1]` 같은 표시로 바꿔 보냅니다. Claude 의 답이 오면 표시를 다시 원래 값으로 바꿔 사용자에게 보여 줍니다. 어떤 표시가 어떤 값이었는지는 요청을 처리하는 동안 서버 메모리에만 두고, 로그에는 저장하지 않습니다.

| 무엇을 | 어떻게 찾나 |
| --- | --- |
| 전화번호, 메일, 사번, 출입증 번호, 차량 번호 | 형식이 정해져 있어 정규식으로 찾습니다 |
| 사람 이름 | 형식이 없어 사내 명부(인사 DB 의 직원, 방문 등록 명단의 방문객)와 대조해 찾습니다 |
| 팀 대표 연락처 | 개인정보가 아니고 답에도 필요하므로 가리지 않습니다 (`public_contacts.json`) |

같은 요청을 마스킹해서 다시 보내 봅시다.

In [ ]:
PHONE_DEFAULT = r'\b0\d{1,2}-\d{3,4}-\d{4}\b'     # 지금 쓰는 전화번호 정규식 (하이픈으로 쓴 번호만 찾습니다)


def make_masker(phone_pattern=PHONE_DEFAULT, add_names=(), add_pass=()):
    """마스킹 설정을 만듭니다. 1-4 에서 세 값을 바꿔 다시 만듭니다"""
    patterns = [('전화번호', phone_pattern, '전화'),
                ('메일', r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+', '메일'),
                ('사번', r'\bHR\d{6}\b', '사번'),
                ('출입증 번호', r'\bVC-\d{4}-\d{4}\b', '출입증'),
                ('차량 번호', r'\b\d{2,3}[가-힣]\d{4}\b', '차량')]
    names = {'직원 명부': list(directory['직원']), '방문 등록 명단': list(directory['방문객']) + list(add_names)}
    keep = list(public_contacts) + list(add_pass)                  # 가리지 않고 그대로 보낼 연락처
    return {'patterns': patterns, 'names': names, 'keep': keep}


def find_personal(text, masker):
    """글에서 개인정보를 찾아 (값, 종류, 찾은 방법) 목록으로 돌려줍니다"""
    found = []
    for kind, pattern, label in masker['patterns']:
        for value in re.findall(pattern, text):
            if value not in masker['keep']:
                found.append((value, kind, label, '정규식'))
    for source, names in masker['names'].items():
        for name in names:
            if name in text:
                found.append((name, '이름', '이름', source))
    return list({item[0]: item for item in found}.values())      # 같은 값은 한 번만


def mask(text, masker):
    """찾은 값을 [이름1], [전화1] 같은 표시로 바꿉니다. (바꾼 글, 표시 → 원래 값 대응표, 찾은 목록)을 돌려줍니다"""
    found = find_personal(text, masker)
    table, counts = {}, defaultdict(int)
    for value, kind, label, how in found:
        counts[label] += 1
        token = f'[{label}{counts[label]}]'
        table[token] = value
        text = text.replace(value, token)
    return text, table, found


def unmask(answer, table):
    """답에 들어 있는 표시를 원래 값으로 되돌립니다"""
    for token, value in table.items():
        answer = answer.replace(token, value)
    return answer


masker = make_masker()
masked_prompt, table, found = mask(prompt, masker)
answer = call_claude(masked_prompt)

print('찾은 개인정보')
for (value, kind, label, how), token in zip(found, table):
    print(f'  {value} ({kind}, {how}) → {token}')
print_box('Claude 에 보낸 프롬프트 (마스킹 뒤)', masked_prompt)
print('Claude 의 답 │', brief(answer, 300))
print('보여 줄 답   │', brief(unmask(answer, table), 300))

### 1-3 · 정답 표시 문장으로 측정하기

요청 한 건이 잘 가려졌다고 해서 마스킹이 잘 동작한다고 볼 수는 없습니다. 가려야 할 값을 미리 표시해 둔 문장 9개(`labeled_sentences.json`)로 마스킹을 측정해 봅시다. **놓친 값**은 가려야 하는데 가려지지 않은 값이고, **잘못 가린 값**은 가리지 않아도 되는데 가린 값입니다.

In [ ]:
def measure(masker, show=True):
    """정답 표시 문장마다 마스킹 결과를 보고, 놓친 값과 잘못 가린 값을 셉니다"""
    missed_all, wrong_all = [], []
    for sentence in labeled:
        masked, table, found = mask(sentence['text'], masker)
        missed = [value for value in sentence['hide'] if value in masked]
        wrong = [value for value, *_ in found if not any(value in answer for answer in sentence['hide'])]
        missed_all += missed
        wrong_all += wrong
        if show:
            mark = '정확' if not missed and not wrong else '확인'
            print(f'[{mark}] {masked}')
            if missed:
                print('        놓친 값     │', ', '.join(missed))
            if wrong:
                print('        잘못 가린 값 │', ', '.join(wrong))
    print(f'\n놓친 값 {len(missed_all)}개, 잘못 가린 값 {len(wrong_all)}개')
    return missed_all, wrong_all


first_missed, first_wrong = measure(make_masker())

### 1-4 · 놓친 원인을 고치고 다시 측정하기

1-3 출력에서 「확인」으로 표시된 문장을 보면 왜 놓쳤는지, 왜 잘못 가렸는지 알 수 있습니다. 아래 셀에서 마스킹 설정 세 가지를 고치고 다시 측정해 봅시다.

In [ ]:
# 1. 전화번호를 찾는 정규식입니다. 지금은 하이픈으로 쓴 번호만 찾습니다.
PHONE_PATTERN = ____
# 예: r'\b0\d{1,2}[-. ]\d{3,4}[-. ]\d{4}\b'   (하이픈, 점, 띄어쓰기로 쓴 번호를 모두 찾습니다)

# 2. 명부에 더할 이름입니다. 지금 명부에는 어제까지 등록된 방문객만 있습니다.
ADD_NAMES = ____
# 예: today_visitors   (오늘 아침 방문 등록 시스템에 올라온 이름 목록)

# 3. 가리지 않고 그대로 보낼 연락처입니다. 지금은 팀 대표 메일 두 개만 있습니다.
ADD_PASS = ____
# 예: ['02-3456-7890']   (총무팀 대표번호)

masker = make_masker(PHONE_PATTERN, ADD_NAMES, ADD_PASS)
missed, wrong = measure(masker)
print(f'처음 설정 → 지금 설정: 놓친 값 {len(first_missed)} → {len(missed)}개, 잘못 가린 값 {len(first_wrong)} → {len(wrong)}개')

## 2단계 · 감사 로그 (예제 1-5)

오늘 하루 들어온 요청 25건을 1단계에서 정한 마스킹으로 처리하면서 감사 로그를 기록해 봅시다. 감사 로그는 나중에 문제가 생겼을 때 무슨 일이 있었는지 추적하는 근거가 됩니다. 하지만 질문 원문까지 저장하면 로그가 또 하나의 유출 경로가 됩니다. 로그에 기록할 항목을 정하고, 로그에 개인정보가 들어갔는지 확인해 봅시다.

In [ ]:
LOG_FIELDS = ____
# 아래 항목 가운데 골라 목록으로 적습니다. 예: ['시각', '계정', '가린 건수']
# 항목: '시각', '계정', '구분', '질문 원문', '가린 질문', '가린 건수', '보낸 문서', '거절 여부'
# 항목이 많을수록 추적하기 쉽지만, 질문이 들어가는 항목에는 개인정보가 함께 기록될 수 있습니다.

REFUSAL_WORDS = ['답할 수 없', '알려 드릴 수 없', '알려드릴 수 없', '제공할 수 없', '공개할 수 없', '확인해 드릴 수 없',
                 '찾지 못', '찾을 수 없', '확인할 수 없', '조회할 수 없', '제공하지 않', '드리기 어렵',
                 '내용이 없', '정보가 없', '권한이 없', '권한 밖']


def handle(request, masker):
    """요청 한 건을 처리하고, 로그에 기록할 수 있는 항목을 모두 돌려줍니다"""
    hits = search(request['question'], request['user'])
    masked_prompt, table, found = mask(build_prompt(request['question'], request['user'], hits), masker)
    answer = call_claude(masked_prompt)
    masked_question = mask(request['question'], masker)[0]
    return {'시각': request['time'], '계정': request['account'], '구분': request['user'],
            '질문 원문': request['question'], '가린 질문': masked_question, '가린 건수': len(found),
            '보낸 문서': [doc['doc_id'] for doc in hits],
            '거절 여부': any(word in answer for word in REFUSAL_WORDS)}


audit_log = []
for request in day_requests:
    record = handle(request, masker)
    audit_log.append({field: record[field] for field in LOG_FIELDS})

print('감사 로그 (앞 5건)')
for row in audit_log[:5]:
    print('  ', json.dumps(row, ensure_ascii=False))

log_text = json.dumps(audit_log, ensure_ascii=False)
left = [value for request in day_requests for value in request['personal'] if value in log_text]
print(f'\n로그에 들어간 개인정보 {len(left)}개', '· ' + ', '.join(left) if left else '')

## 3단계 · 모니터링 (예제 1-5)

감사 로그를 사용자 계정별로 모아, 이상 기록이 있는 사용자를 찾아봅시다.

요청 하나하나는 챗봇이 잘 거절했더라도, 같은 계정이 하루 동안 볼 수 없는 정보를 여러 번 물었다면 따로 확인해 볼 필요가 있습니다. 하루 거절 수가 몇 번 이상인 계정을 따로 표시할지 정하고, 어느 계정이 표시되는지 확인해 봅시다.

In [ ]:
ALERT_REFUSALS = ____
# 1, 3, 7 중에서 고릅니다. 하루 거절 수가 이 값 이상인 계정을 따로 표시합니다.
# 낮으면 평범한 사용자도 함께 표시되고, 높으면 반복해서 시도하는 계정을 놓칠 수 있습니다.

if not {'계정', '거절 여부'} <= set(LOG_FIELDS):
    print('로그에 「계정」과 「거절 여부」 항목이 없어 계정별로 모을 수 없습니다. 2단계의 LOG_FIELDS 를 다시 정해 봅시다')
else:
    summary = defaultdict(lambda: {'요청': 0, '거절': 0, '가린 건수': 0})
    for row in audit_log:
        summary[row['계정']]['요청'] += 1
        summary[row['계정']]['거절'] += row['거절 여부']
        summary[row['계정']]['가린 건수'] += row.get('가린 건수', 0)
    print('계정별 집계')
    for account, counts in sorted(summary.items()):
        alert = '  ← 확인 대상' if counts['거절'] >= ALERT_REFUSALS else ''
        print(f"  {account:<11} 요청 {counts['요청']:>2}건, 거절 {counts['거절']}건, 가린 개인정보 {counts['가린 건수']}개{alert}")

    flagged = [account for account, counts in summary.items() if counts['거절'] >= ALERT_REFUSALS]
    for account in flagged:
        print(f'\n■ {account} 의 거절된 요청 (로그에 기록한 항목만 보입니다)')
        for row in audit_log:
            if row['계정'] == account and row['거절 여부']:
                print('  ', json.dumps({key: value for key, value in row.items() if key != '계정'}, ensure_ascii=False))

**마무리** · 1, 2, 3단계의 값을 채운 채로 [런타임] → [모두 실행] 으로 한 번 더 실행하고, 출력이 보이는 상태로 저장합니다.

## 확장 미션(선택) · 내 업무에 마스킹, 감사 로그, 모니터링 설계해 보기

내 업무에서 생성형 AI 에 개인정보나 사내 정보를 보내는 곳이 있는지 찾아보고, 이번 미션처럼 마스킹, 감사 로그, 모니터링을 설계해 봅시다.

1. 내 업무에서 AI 에 보내는 글(질문, 문서, 기록 등)에 어떤 정보가 들어가는지 적습니다.
2. 마스킹: 가려야 할 값과 찾는 방법(형식, 명부 등), 가리지 않을 값을 정합니다.
3. 감사 로그: 기록할 항목과 기록하지 않을 항목을 정합니다.
4. 모니터링: 로그에서 무엇을 모아 보고, 몇 번부터 따로 확인할지 정합니다.

**설계:** (이 셀을 두 번 클릭해 적습니다)